# Reasoning Memory: запуск из GitHub
Включите GPU runtime. Первая ячейка клонирует или обновляет проект в `/content/reasoning-memory`.
Для приватного репозитория добавьте в Colab Secrets секрет `GITHUB_TOKEN` и разрешите доступ
этому ноутбуку. Достаточно fine-grained PAT с **Contents: Read-only** для `Ferraronp/reasoning-memory`.
Без Secrets ячейка предложит скрытый ввод. Токен не записывается в URL remote или файлы проекта.
Повторный запуск подтягивает `main` через fast-forward; локальные правки исходников останавливают обновление.
После обновления выполните ячейку установки. Результаты в `runs/` сохраняются в пределах текущей сессии.
Обновление файлов на диске не обновляет уже открытые ячейки ноутбука: при изменении самого ноутбука
откройте его свежую версию из GitHub/Colab.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import tempfile
from getpass import getpass
from google.colab import files, userdata

REPO_URL = 'https://github.com/Ferraronp/reasoning-memory.git'
BRANCH = 'main'
PROJECT_ROOT = Path('/content/reasoning-memory')

def sync_project(repo_url, branch, project_root, git_env=None):
    project_root = Path(project_root)
    def git(*args, cwd=None):
        result = subprocess.run(['git', '-c', 'credential.helper=', *args], cwd=cwd,
                                env=git_env, capture_output=True, text=True)
        if result.returncode:
            raise RuntimeError(result.stderr.strip() or result.stdout.strip() or 'Git failed')
        return result.stdout.strip()
    if not project_root.exists():
        git('clone', '--branch', branch, '--single-branch', repo_url, str(project_root))
    else:
        if not (project_root / '.git').is_dir():
            raise RuntimeError(f'{project_root} exists but is not a git clone. Choose another PROJECT_ROOT.')
        origin = git('remote', 'get-url', 'origin', cwd=project_root)
        if origin.rstrip('/') != repo_url.rstrip('/'):
            raise RuntimeError('The existing folder has a different origin; choose another PROJECT_ROOT.')
        if git('branch', '--show-current', cwd=project_root) != branch:
            raise RuntimeError(f'The checkout is not on {branch}. Switch it manually before updating.')
        if git('status', '--porcelain', '--untracked-files=no', cwd=project_root):
            raise RuntimeError('Local source changes found. Commit/stash them before updating; nothing was overwritten.')
        git('fetch', 'origin', branch, cwd=project_root)
        git('merge', '--ff-only', 'FETCH_HEAD', cwd=project_root)
    print('Project:', project_root)
    print('Commit:', git('log', '-1', '--format=%h %s', cwd=project_root))

def sync_private_project():
    try:
        token = userdata.get('GITHUB_TOKEN')
    except Exception:
        token = None
    if not token:
        token = getpass('GitHub token (read-only Contents for this repository): ').strip()
    if not token:
        raise ValueError('A token is required for this private repository')
    # Helper contains only environment-variable references, never the actual token.
    with tempfile.TemporaryDirectory(prefix='rm-git-auth-') as auth_dir:
        askpass = Path(auth_dir) / 'askpass.sh'
        askpass.write_text('#!/bin/sh\ncase "$1" in\n  *Username*) printf "%s\\n" "x-access-token" ;;\n  *) printf "%s\\n" "$RM_GITHUB_TOKEN" ;;\nesac\n')
        askpass.chmod(0o700)
        env = os.environ.copy()
        env.update(GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT='0', RM_GITHUB_TOKEN=token)
        try:
            sync_project(REPO_URL, BRANCH, PROJECT_ROOT, env)
        finally:
            env.pop('RM_GITHUB_TOKEN', None)
            token = None

sync_private_project()
assert (PROJECT_ROOT / 'pyproject.toml').is_file()


## Установка и проверка окружения
INT8 пока выключен. Установка может занять несколько минут.


In [ ]:
def run_process(argv):
    # Pipes are read in Python so Jupyter/Colab reliably displays child output.
    with subprocess.Popen(argv, cwd=PROJECT_ROOT, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        return process.wait()

if run_process([sys.executable, '-m', 'pip', 'install', '-e', f'{PROJECT_ROOT}[hf]']) != 0:
    raise RuntimeError('Installation failed; see output above')

def run_cli(*args):
    code = run_process([sys.executable, '-u', '-m', 'reasoning_memory', *map(str, args)])
    if code not in (0, 2):
        raise RuntimeError(f'Ошибка запуска, exit={code}; см. вывод выше')
    if code == 2:
        print('Запуск закончен с диагностическими ошибками. Причина и текст генерации показаны выше.')
    # Intentionally no CompletedProcess return value.

run_cli('doctor')
if run_process([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v']) != 0:
    raise RuntimeError('Tests failed')


## Проверка контроллера без модели
Должны получиться full и compact с ответом 5.

Здесь также используется guided_single, но с искусственным backend без весов.


In [ ]:
run_cli('pair', '--config', 'configs/mock_guided.json', '--tasks', 'data/smoke.jsonl')


## Реальный запуск: guided_single
Начните с 0.6B FP16. Контроллер открывает эксперимент, затем summary и final answer;
модель генерирует их содержание. Это техническая проверка с заданной структурой,
не тест самостоятельной разметки. Для автономного режима используйте исходный
`configs/qwen3_06b_fp16.json` с `protocol: "autonomous"`.
Первый эксперимент и summary общие для full/compact. Затем сравнивается продолжение.
9B пока не настроена: нужен конкретный checkpoint и проверка адаптера.


In [ ]:
CONFIG = json.loads((PROJECT_ROOT / 'configs/qwen3_06b_fp16_guided.json').read_text())
CONFIG.update(dtype='float16', quantization='none', allow_restore=False, seed=42)
# CONFIG['max_new_tokens'] = 1536  # При обрыве эксперимента изучить trace и пересмотреть бюджет.
CONFIG_PATH = PROJECT_ROOT / 'configs/session.json'
CONFIG_PATH.write_text(json.dumps(CONFIG, indent=2))
print(json.dumps(CONFIG, indent=2))


In [ ]:
run_cli('pair', '--config', CONFIG_PATH, '--tasks', 'data/smoke.jsonl', '--limit', 1)


## Два этапа: использование сохранённого вывода
Первый этап общий. После его вывода создаются full и compact; только затем обе ветки
получают второй этап. Модель рассуждает снова, пишет второй вывод и финальный ответ.
Начните с LIMIT = 1 (ожидаемый ответ 17), затем поставьте 3 (17, 25, 53).
Должно быть experiments: 2 в каждой завершённой ветке. Строгая проверка требует ровно число.
Это диагностические примеры: исходное условие остаётся, поэтому модель может пересчитать результат.


In [ ]:
TWO_STAGE_CONFIG = json.loads((PROJECT_ROOT / 'configs/qwen3_17b_fp16_two_stage.json').read_text())
# Здесь можно менять модель, бюджеты и seed независимо от предыдущего smoke.
TWO_STAGE_PATH = PROJECT_ROOT / 'configs/session.json'
TWO_STAGE_PATH.write_text(json.dumps(TWO_STAGE_CONFIG, indent=2))
LIMIT = 1  # Затем 3 — все три диагностические задачи
run_cli('pair', '--config', TWO_STAGE_PATH, '--tasks', 'data/two_stage.jsonl', '--limit', LIMIT)


## Сравнение способов подачи на 1.7B FP16
Предыдущая ячейка теперь запускает 1.7B с прежним способом подачи внутри think.
Ячейка ниже запускает те же задачи, seed и бюджеты, но второй этап подаётся
отдельным пользовательским сообщением. Вывод первого этапа становится ответом
ассистента; full дополнительно сохраняет его reasoning, compact — удаляет.
Это диагностика с двумя thinking-блоками, не целевой автономный режим.
Сначала LIMIT=1 в обеих ячейках, затем при необходимости LIMIT=3.


In [ ]:
run_cli('pair', '--config', 'configs/qwen3_17b_fp16_chat_two_stage.json',
        '--tasks', 'data/two_stage.jsonl', '--limit', LIMIT)


## Посмотреть последний запуск
Протокольная ошибка — диагностический результат; она не означает опровержение гипотезы.
После успешного smoke можно заменить `data/smoke.jsonl` на `data/dev_examples.jsonl`.


In [ ]:
latest = max((PROJECT_ROOT / 'runs').iterdir(), key=lambda p: p.name)
print('Run:', latest)
for filename in ['manifest.json', 'results.json', 'summary.json']:
    path = latest / filename
    if path.exists():
        print(filename, path.read_text()[:12000])
# Полные входы и выходы: latest / 'task_00000/events.jsonl'


## Скачать результаты до завершения сессии
Временный диск Colab не является постоянным хранилищем.


In [ ]:
import shutil
result_zip = shutil.make_archive('/content/reasoning-memory-results', 'zip', PROJECT_ROOT, 'runs')
files.download(result_zip)
